## Import Cleaned Data and Merge Latest Data

**Steps**:
- Import various trade files
- Change formatting into per trade instead of daily settlement (keeping both clean versions though)
- Plot each quarter series individually (with faded market data if necessary)
- Review the statistics of successful and unsuccessful trades (advise potential stop-loss positions)

In [18]:
# Libraries
import pandas as pd 
import numpy as np 
import statistics as stats 
import matplotlib.pyplot as plt 
import math 

# Need to reindex to trading days 
import pandas_market_calendars as mcal

In [21]:
# Import new data a reformat ready for merging
df = pd.read_csv(r".\Trade_Data\Trade_data_update.csv", engine="python")

# Rename columns for easier handling
cols = [
    "Date", "Stamp", "ID", 
    "ES_signal", "ES_Open_Price", 
    "NQ_signal", "NQ_Open_Price", 
    "ZN_signal", "ZN_Open_Price", 
    "FGBL_signal", "FGBL_Open_Price", 
    "6E_signal", "6E_Open_Price", 
    "6J_signal", "6J_Open_Price"
    ] 
df.columns = cols 

# Drop unnecessary columns and add quarter columns
df["Quarter"] = "2026Q2" 
cols = [
    "Date", "Quarter",
    "ES_signal", "ES_Open_Price", 
    "NQ_signal", "NQ_Open_Price", 
    "ZN_signal", "ZN_Open_Price", 
    "FGBL_signal", "FGBL_Open_Price", 
    "6E_signal", "6E_Open_Price", 
    "6J_signal", "6J_Open_Price"
    ]
df = df[cols]

# Handle index 
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True) 
df["Date"] = df["Date"] + pd.offsets.BDay(1) 
df = df.dropna(subset=["Date"])
df = df.set_index("Date") 
df = df.sort_index()

# Change string signal to polar signals 
mapping = {np.nan:0, "BULLISH": 1, "BEARISH": -1, "ENDOFSERIES":0}
tickers = ["ES", "NQ", "ZN", "FGBL", "6E", "6J"]
for tick in tickers:
    df[f"{tick}_signal"] = df[f"{tick}_signal"].map(mapping) 
    df[f"{tick}_signal"] = df[f"{tick}_signal"].ffill().fillna(0).astype(int)
    df[f"{tick}_Open_Price"] = df[f"{tick}_Open_Price"].ffill().astype(float) # Only impacting last traded day

# Fix calendar
cme = mcal.get_calendar("CME_Equity")
schedule = cme.schedule(start_date=df.index.min(),
                        end_date=df.index.max())
trading_days = schedule.index
df = df.reindex(trading_days)

# Isolate new data from 2026-06-03
df = df.loc["2026-06-03":]

# Save to csv for use 
df.reset_index().to_csv(r".\Trade_Data\Trade_data_update_cleaned.csv", index=False)

# Preview dataframe 
df 

,Quarter,ES_signal,ES_Open_Price,NQ_signal,NQ_Open_Price,ZN_signal,ZN_Open_Price,FGBL_signal,FGBL_Open_Price,6E_signal,6E_Open_Price,6J_signal,6J_Open_Price
2026-06-03,2026Q2,1,7628.00,-1,30741.75,-1,109.765625,1,126.0,-1,1.16365,1,0.006262
2026-06-04,2026Q2,1,7539.75,1,30453.00,-1,109.390625,0,125.5,-1,1.16040,-1,0.006254
2026-06-05,2026Q2,1,7589.50,1,30414.00,-1,109.593750,0,125.5,-1,1.16170,-1,0.006255
2026-06-08,2026Q2,1,7368.00,1,28844.50,-1,109.046875,0,125.5,-1,1.15195,-1,0.006243
2026-06-09,2026Q2,1,7412.75,1,29435.00,-1,108.953125,0,125.5,-1,1.15385,-1,0.006247
2026-06-10,2026Q2,-1,7380.00,1,29094.75,1,109.234375,0,125.5,-1,1.15435,-1,0.006237
2026-06-11,2026Q2,1,7259.25,1,28455.00,1,109.015625,0,125.5,0,1.15375,0,0.006230
2026-06-12,2026Q2,-1,7397.50,1,29456.75,1,109.671875,0,125.5,0,1.15375,0,0.006230
2026-06-15,2026Q2,-1,7480.00,1,29907.00,1,109.781250,0,125.5,0,1.15375,0,0.006230
2026-06-16,2026Q2,0,7480.00,0,29907.00,0,109.781250,0,125.5,0,1.15375,0,0.006230


**Import both dataframes and merge for good copy**

In [43]:
# Old dataframe 
df_old = pd.read_csv(r".\Trade_Data\Trade_data.csv", engine="python", index_col=0, parse_dates=True) 

# New dataframe 
df_new = pd.read_csv(r".\Trade_Data\Trade_data_update_cleaned.csv", engine="python", index_col=0, parse_dates=True) 

# Create final dataframe 
df = df_old.copy() 

# Ready new dataframe for merging
df_new[["FGBL_signal", "FGBL_Open_Price"]] = 0
df_new = df_new.rename(columns={"FGBL_signal": "FGBM_signal", "FGBL_Open_Price": "FGBM_Open_Price"})
df_new = df_new.drop(["6J_signal", "6J_Open_Price"], axis=1)

# Merge 
df.loc[df_new.index.min():] = df_new 

# Save for use 
df.reset_index().to_csv(r".\Trade_Data\Trade_data_all.csv", index=False)

**Import and process trade data into new 'per trade' appoarch**:

In [45]:
# Import data 
df = pd.read_csv(r".\Trade_Data\Trade_data_all.csv", engine="python", parse_dates=True, index_col=0) 

# Preview data 
df.head() 

,Quarter,ES_signal,ES_Open_Price,NQ_signal,NQ_Open_Price,ZN_signal,ZN_Open_Price,FGBM_signal,FGBM_Open_Price,6E_signal,6E_Open_Price
index,,,,,,,,,,,
2025-06-23,2025Q3,1,5964.00,0,NaN,0,NaN,1,117.79,-1,1.15425
2025-06-24,2025Q3,1,6078.00,0,NaN,0,NaN,1,117.83,-1,1.16400
2025-06-25,2025Q3,1,6144.75,0,NaN,0,NaN,1,117.60,1,1.15586
2025-06-26,2025Q3,1,6144.75,0,NaN,0,NaN,1,117.75,1,1.17190
2025-06-27,2025Q3,1,6197.50,0,NaN,0,NaN,1,117.81,1,1.17610


In [73]:
# Function for analysing trading record 
def create_trade_record(df, tick): 

    # Log
    trades = [] 

    # Assign series 
    signal_col = f"{tick}_signal" 
    price_col = f"{tick}_Open_Price"

    # Initialize 
    current_signal = 0
    open_date = None 
    close_date = None 
    trade_no = 1 
    open_idx = None

    for date, row in df.iterrows(): 

        signal = row[signal_col] 
        price = row[price_col] 

        # Skip missing signals
        if pd.isna(signal):
            continue 

        # No position currently 
        if current_signal == 0: 

            if signal != 0:
                current_signal = signal 
                open_date = date 
                open_price = price 
                open_quarter = row["Quarter"]
                open_idx = df.index.get_loc(date)

        # Currently in a position 
        else:

            # Close only 
            if signal == 0: 

                close_idx = df.index.get_loc(date) 

                trades.append({
                    "Trade No": trade_no,
                    "Quarter": open_quarter,
                    "Open Date": open_date, 
                    "Open Price": open_price,
                    "Close Date": date,
                    "Close Price": price, 
                    "Trading Days": close_idx - open_idx,
                    "Type": "long" if current_signal == 1 else "short",
                    "Return": (
                        (price / open_price - 1)
                        if current_signal == 1
                        else (open_price / price -1)
                    ),
                    "Log_Return": (
                        np.log(price / open_price)
                        if current_signal == 1 
                        else np.log(open_price / price)
                    ),
                    "Market_Return": (
                        (price / open_price - 1)
                    ),
                    "Market_Log_Return": (
                        np.log(price / open_price)
                    )
                })

                trade_no += 1 
                current_signal = 0 
            
            # Reverse position 
            elif signal != current_signal: 

                close_idx = df.index.get_loc(date)

                # Close old trade 
                trades.append({
                    "Trade No": trade_no,
                    "Quarter": open_quarter,
                    "Open Date": open_date, 
                    "Open Price": open_price,
                    "Close Date": date,
                    "Close Price": price, 
                    "Trading Days": close_idx - open_idx,
                    "Type": "long" if current_signal == 1 else "short",
                    "Return": (
                        (price / open_price - 1)
                        if current_signal == 1
                        else (open_price / price -1)
                    ),
                    "Log_Return": (
                        np.log(price / open_price)
                        if current_signal == 1 
                        else np.log(open_price / price)
                    ),
                    "Market_Return": (
                        (price / open_price - 1)
                    ),
                    "Market_Log_Return": (
                        np.log(price / open_price)
                    )
                })

                trade_no += 1 

                # Open new trade immediately 
                current_signal = signal 
                open_date = date 
                open_price = price 
                open_quarter = row["Quarter"]
                open_idx = df.index.get_loc(date)

    # Leave final trade open if necessary
    if current_signal != 0:
        trades.append({
            "Trade No": trade_no,
            "Quarter": open_quarter,
            "Open Date": open_date,
            "Open Price": open_price,
            "Close Date": pd.NaT,
            "Close Price": None,
            "Trading Days": None,
            "Type": "long" if current_signal == 1 else "short",
            "Return": (
                (price / open_price - 1)
                if current_signal == 1
                else (open_price / price -1)
            ),
            "Log_Return": (
                np.log(price / open_price)
                if current_signal == 1 
                else np.log(open_price / price)
            ),
            "Market_Return": (
                (price / open_price - 1)
            ),
            "Market_Log_Return": (
                np.log(price / open_price)
            )
        })

    return pd.DataFrame(trades).set_index(["Trade No"]) 

# Tickers 
tickers = ["ES", "NQ", "ZN", "FGBM", "6E"]

# Create trade data 
all_trades = []
for ticker in tickers:
    trades = create_trade_record(df, ticker)
    trades.insert(0, "Ticker", ticker) 

    # Compute cum log returns
    trades["Cum_Log_Return"] = trades["Log_Return"].cumsum()
    trades["Cum_Market_Log_Return"] = trades["Market_Log_Return"].cumsum()

    # Add equity curves
    trades["Equity_Curve"] = np.exp(trades["Cum_Log_Return"])
    trades["Market_Equity_Curve"] = np.exp(trades["Cum_Market_Log_Return"])

    all_trades.append(trades)

# Compile and preview
all_trades = pd.concat(all_trades, ignore_index=True)
all_trades

,Ticker,Quarter,Open Date,Open Price,Close Date,Close Price,Trading Days,Type,Return,Log_Return,Market_Return,Market_Log_Return,Cum_Log_Return,Cum_Market_Log_Return,Equity_Curve,Market_Equity_Curve
0,ES,2025Q3,2025-06-23,5964.00000,2025-07-07,6307.75000,10,long,0.057637,0.056038,0.057637,0.056038,0.056038,0.056038,1.057637,1.057637
1,ES,2025Q3,2025-07-07,6307.75000,2025-07-14,6274.00000,5,short,0.005379,0.005365,-0.005351,-0.005365,0.061403,0.050673,1.063327,1.051979
2,ES,2025Q3,2025-07-14,6274.00000,2025-09-05,6515.00000,39,long,0.038412,0.037693,0.038412,0.037693,0.099096,0.088366,1.104172,1.092388
3,ES,2025Q3,2025-09-05,6515.00000,2025-09-08,6483.75000,1,short,0.004820,0.004808,-0.004797,-0.004808,0.103904,0.083558,1.109494,1.087148
4,ES,2025Q3,2025-09-08,6483.75000,2025-09-17,6669.50000,7,long,0.028649,0.028246,0.028649,0.028246,0.132150,0.111803,1.141279,1.118293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,6E,2026Q2,2026-04-29,1.17385,2026-04-30,1.17005,1,short,0.003248,0.003242,-0.003237,-0.003242,-0.039676,-0.003744,0.961101,0.996263
126,6E,2026Q2,2026-04-30,1.17005,2026-05-05,1.17155,3,long,0.001282,0.001281,0.001282,0.001281,-0.038394,-0.002463,0.962333,0.997540
127,6E,2026Q2,2026-05-05,1.17155,2026-05-06,1.17160,1,short,-0.000043,-0.000043,0.000043,0.000043,-0.038437,-0.002420,0.962292,0.997582
128,6E,2026Q2,2026-05-06,1.17160,2026-05-19,1.16735,9,long,-0.003628,-0.003634,-0.003628,-0.003634,-0.042071,-0.006055,0.958802,0.993964


**Analyse each ticker**